In [1]:
import os, torch
print("CUDA_VISIBLE_DEVICES =", os.environ.get("CUDA_VISIBLE_DEVICES"))
print("LD_LIBRARY_PATH =", os.environ.get("LD_LIBRARY_PATH"))
print("cuda available =", torch.cuda.is_available())
print("device count =", torch.cuda.device_count())
if torch.cuda.is_available():
    print("device 0 =", torch.cuda.get_device_name(0))

CUDA_VISIBLE_DEVICES = 1
LD_LIBRARY_PATH = None
cuda available = True
device count = 1
device 0 = NVIDIA RTX A6000


In [1]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:128,expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import math
import time
import json
import random
import csv
import gc
from dataclasses import dataclass
from typing import List, Dict, Any, Tuple, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from torch import amp

from h_lcm1_Copy1 import HyperbolicLCM


# ============================================================
# CONFIG
# ============================================================

@dataclass
class GRPOConfig:
    datasets_to_run: Tuple[str, ...] = ("boolq_local",)

    # LOCAL BOOLQ PARQUET FILES
    hf_train_file: str = "/home/user/twovolume/Nisha/Finetune/boolq/train-00000-of-00001.parquet"
    hf_validation_file: str = "/home/user/twovolume/Nisha/Finetune/boolq/validation-00000-of-00001.parquet"

    out_dir: str = "runs/hlcm_boolq_local_grpo_better"
    cache_dir: str = "boolq_local_cached_features_grpo_better"
    ref_logits_dir: str = "boolq_local_cached_ref_logits_grpo_better"

    # pretrained hlcm
    ckpt_path: str = "runs/hyperbolic_cluster/checkpoints/ckpt_best.pt"
    normalizer_path: str = "normalizer.pt"

    # conceptizer
    encoder_name: str = "microsoft/deberta-v3-small"
    chunk_tok_len: int = 256
    seq_len: int = 12
    encoder_batch_size: int = 16
    conceptizer_device: str = "cpu"

    # hlcm arch
    in_dim: int = 768
    model_dim: int = 4096
    num_heads: int = 32
    num_layers: int = 12
    ffn_mult: int = 4
    dropout: float = 0.30
    manifold_c: float = 0.002
    causal: bool = True
    input_scale: float = 0.05
    input_max_norm: float = 1.0

    # finetune mode
    finetune_mode: str = "last_blocks"   # "last_blocks" | "full"
    n_last_blocks: int = 2

    # loader / memory
    train_batch_size: int = 2
    eval_batch_size: int = 4
    grad_accum_steps: int = 8
    num_workers: int = 0

    # SFT
    sft_epochs: int = 2
    sft_lr: float = 3e-5
    sft_warmup_ratio: float = 0.06

    # GRPO
    grpo_epochs: int = 1
    grpo_lr: float = 8e-6
    grpo_warmup_ratio: float = 0.06

    # optimization
    weight_decay: float = 0.01
    max_grad_norm: float = 1.0
    use_bf16: bool = True

    # logits / regularization
    mcq_logit_temperature: float = 0.2
    label_smoothing: float = 0.02
    use_class_weights: bool = True

    # GRPO policy / reward
    grpo_policy_temperature: float = 0.8
    grpo_group_size: int = 4
    grpo_beta_kl: float = 0.03
    entropy_bonus: float = 0.002

    reward_correct: float = 1.0
    reward_incorrect: float = 0.0
    use_group_relative_advantage: bool = True

    # misc
    min_valid_choices: int = 2
    seed: int = 42
    prefer_gpu_index: int = 0


# ============================================================
# BOOLQ TEMPLATES
# ============================================================

BOOLQ_YES_TEMPLATES = [
    "Answer: yes",
    "The correct answer is yes.",
    "Based on the passage, the answer is yes.",
    "The statement is supported by the passage.",
]

BOOLQ_NO_TEMPLATES = [
    "Answer: no",
    "The correct answer is no.",
    "Based on the passage, the answer is no.",
    "The statement is not supported by the passage.",
]


# ============================================================
# UTILS
# ============================================================

def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


def pick_device(prefer_gpu_index: int = 0) -> torch.device:
    if not torch.cuda.is_available():
        return torch.device("cpu")
    n = torch.cuda.device_count()
    idx = 0 if n == 1 else max(0, min(prefer_gpu_index, n - 1))
    torch.cuda.set_device(idx)
    return torch.device(f"cuda:{idx}")


def build_amp_dtype(device: torch.device) -> torch.dtype:
    if device.type != "cuda":
        return torch.float32
    return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16


def save_checkpoint(payload: dict, path: str):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    torch.save(payload, path)


def append_dict_to_csv(path: str, row: Dict[str, Any]):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    file_exists = os.path.isfile(path)
    with open(path, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(row.keys()))
        if not file_exists:
            writer.writeheader()
        writer.writerow(row)


def write_single_row_csv(path: str, row: Dict[str, Any]):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(row.keys()))
        writer.writeheader()
        writer.writerow(row)


def gpu_mem_mb(device: Optional[torch.device] = None) -> Dict[str, float]:
    if not torch.cuda.is_available():
        return {"alloc_mb": 0.0, "reserved_mb": 0.0, "max_alloc_mb": 0.0}
    if device is None:
        idx = torch.cuda.current_device()
    else:
        idx = device.index if device.index is not None else torch.cuda.current_device()
    return {
        "alloc_mb": torch.cuda.memory_allocated(idx) / (1024 ** 2),
        "reserved_mb": torch.cuda.memory_reserved(idx) / (1024 ** 2),
        "max_alloc_mb": torch.cuda.max_memory_allocated(idx) / (1024 ** 2),
    }


def cuda_cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def load_normalizer(normalizer_path: str, device: torch.device):
    if not normalizer_path or not os.path.exists(normalizer_path):
        return None, None
    obj = torch.load(normalizer_path, map_location="cpu")
    mu = obj["mu"].float().to(device)
    sigma = obj["sigma"].float().clamp_min(1e-4).to(device)
    return mu, sigma


def clone_state_dict_to_cpu(model: nn.Module) -> Dict[str, torch.Tensor]:
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}


# ============================================================
# FREEZE / UNFREEZE
# ============================================================

def set_requires_grad(m: nn.Module, flag: bool):
    for p in m.parameters():
        p.requires_grad = flag


def freeze_all(model: nn.Module):
    set_requires_grad(model, False)


def unfreeze_all(model: nn.Module):
    set_requires_grad(model, True)


def unfreeze_last_blocks_hlcm(model: HyperbolicLCM, n_last: int):
    freeze_all(model)

    if not hasattr(model, "layers"):
        raise AttributeError("HyperbolicLCM is expected to have attribute 'layers'.")

    layers = list(model.layers)
    if len(layers) == 0:
        raise ValueError("HyperbolicLCM has no layers.")
    n_last = max(1, min(n_last, len(layers)))

    for blk in layers[-n_last:]:
        set_requires_grad(blk, True)

    for name, module in model.named_children():
        if name not in ("layers",):
            set_requires_grad(module, True)


def apply_finetune_mode_hlcm(model: HyperbolicLCM, mode: str, n_last: int):
    if mode == "last_blocks":
        unfreeze_last_blocks_hlcm(model, n_last)
    elif mode == "full":
        unfreeze_all(model)
    else:
        raise ValueError(f"Unknown finetune_mode: {mode}")


# ============================================================
# CONCEPTIZER
# ============================================================

class DebertaConceptizer:
    def __init__(
        self,
        model_name: str,
        chunk_tok_len: int,
        seq_len: int,
        batch_size: int,
        device: torch.device,
    ):
        self.model_name = model_name
        self.chunk_tok_len = int(chunk_tok_len)
        self.seq_len = int(seq_len)
        self.batch_size = int(batch_size)
        self.device = device

        self.tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        self.enc = AutoModel.from_pretrained(model_name).to(device)
        self.enc.eval()

        for p in self.enc.parameters():
            p.requires_grad = False

        self.hidden_size = int(self.enc.config.hidden_size)

    @torch.inference_mode()
    def _embed_chunk_texts(self, chunk_texts: List[str]) -> torch.Tensor:
        if len(chunk_texts) == 0:
            return torch.empty(0, self.hidden_size, dtype=torch.float32)

        inputs = self.tok(
            chunk_texts,
            padding=True,
            truncation=True,
            max_length=self.chunk_tok_len,
            return_tensors="pt",
        )
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        if self.device.type == "cuda":
            with amp.autocast(device_type="cuda", enabled=True, dtype=build_amp_dtype(self.device)):
                out = self.enc(**inputs).last_hidden_state[:, 0, :]
        else:
            out = self.enc(**inputs).last_hidden_state[:, 0, :]

        return out.detach().cpu().float()

    def _pack_into_chunk_texts(self, text: str) -> List[str]:
        ids = self.tok(text, add_special_tokens=False)["input_ids"]
        if not ids:
            return []

        out = []
        for i in range(0, len(ids), self.chunk_tok_len):
            out.append(self.tok.decode(ids[i:i + self.chunk_tok_len], clean_up_tokenization_spaces=True))
            if len(out) >= self.seq_len:
                break
        return out[:self.seq_len]

    @torch.inference_mode()
    def encode_text_fixed(self, text: str) -> Tuple[torch.Tensor, torch.Tensor]:
        chunk_texts = self._pack_into_chunk_texts(text)

        seq = torch.zeros(self.seq_len, self.hidden_size, dtype=torch.float32)
        pad = torch.ones(self.seq_len, dtype=torch.bool)

        if not chunk_texts:
            return seq, pad

        vecs = []
        for i in range(0, len(chunk_texts), self.batch_size):
            vecs.append(self._embed_chunk_texts(chunk_texts[i:i + self.batch_size]))

        vecs = torch.cat(vecs, dim=0)
        c = min(vecs.size(0), self.seq_len)
        seq[:c] = vecs[:c]
        pad[:c] = False
        return seq, pad


# ============================================================
# BOOLQ LOCAL PARQUET LOADING / NORMALIZATION
# ============================================================

def load_boolq_local(cfg: GRPOConfig):
    if not os.path.exists(cfg.hf_train_file):
        raise FileNotFoundError(f"Train parquet not found: {cfg.hf_train_file}")
    if not os.path.exists(cfg.hf_validation_file):
        raise FileNotFoundError(f"Validation parquet not found: {cfg.hf_validation_file}")

    raw = load_dataset(
        "parquet",
        data_files={
            "train": cfg.hf_train_file,
            "validation": cfg.hf_validation_file,
        },
    )

    train_split = raw["train"]
    eval_split = raw["validation"]
    return train_split, eval_split


def normalize_boolq_example(ex: Dict[str, Any], min_valid_choices: int):
    passage = str(ex.get("passage", "")).strip()
    question = str(ex.get("question", "")).strip()
    answer = ex.get("answer", False)

    if question and not question.endswith("?"):
        question = question + "?"

    q_text = (
        f"Task: Answer the yes/no question using the passage.\n"
        f"Passage: {passage}\n"
        f"Question: {question}"
    )

    choice_texts = ["no", "yes"]
    if len(choice_texts) < min_valid_choices:
        choice_texts = ["no", "yes"]

    label = 1 if bool(answer) else 0
    return q_text, choice_texts, label, passage, question


# ============================================================
# CACHED FEATURE BUILD
# ============================================================

def cache_file_path(cfg: GRPOConfig, dataset_name: str, split_name: str) -> str:
    safe_ds = dataset_name.replace("/", "_")
    ensure_dir(cfg.cache_dir)
    return os.path.join(
        cfg.cache_dir,
        f"{safe_ds}_{split_name}_tok{cfg.chunk_tok_len}_seq{cfg.seq_len}.pt"
    )


def ref_logits_file_path(cfg: GRPOConfig, dataset_name: str, split_name: str) -> str:
    safe_ds = dataset_name.replace("/", "_")
    ensure_dir(cfg.ref_logits_dir)
    return os.path.join(
        cfg.ref_logits_dir,
        f"{safe_ds}_{split_name}_ref_logits_tok{cfg.chunk_tok_len}_seq{cfg.seq_len}.pt"
    )


def build_or_load_cached_split(
    cfg: GRPOConfig,
    dataset_name: str,
    split_name: str,
    hf_split,
    conceptizer: DebertaConceptizer,
) -> List[Dict[str, Any]]:
    path = cache_file_path(cfg, dataset_name, split_name)

    if hf_split is None:
        return []

    if os.path.exists(path):
        print(f"[cache] loading {path}")
        return torch.load(path)

    print(f"[cache] building {dataset_name} / {split_name}")
    rows = []
    skipped = 0

    for ex in tqdm(hf_split, desc=f"cache:{dataset_name}:{split_name}"):
        try:
            q_text, choice_texts, label, passage, question = normalize_boolq_example(ex, cfg.min_valid_choices)

            q_seq, q_pad = conceptizer.encode_text_fixed(q_text)

            yes_variants = [
                f"Passage: {passage}\nQuestion: {question}\n{t}"
                for t in BOOLQ_YES_TEMPLATES
            ]
            no_variants = [
                f"Passage: {passage}\nQuestion: {question}\n{t}"
                for t in BOOLQ_NO_TEMPLATES
            ]

            all_choice_groups = []
            all_mask_groups = []

            for variants in [no_variants, yes_variants]:
                seqs, pads = [], []
                for txt in variants:
                    s, p = conceptizer.encode_text_fixed(txt)
                    seqs.append(s)
                    pads.append(p)
                all_choice_groups.append(torch.stack(seqs, dim=0))   # [V, T, D]
                all_mask_groups.append(torch.stack(pads, dim=0))     # [V, T]

            rows.append({
                "q": q_seq,
                "qmask": q_pad,
                "choices": torch.stack(all_choice_groups, dim=0),    # [2, V, T, D]
                "cmask": torch.stack(all_mask_groups, dim=0),        # [2, V, T]
                "choice_mask": torch.tensor([True, True], dtype=torch.bool),
                "label": int(label),
                "num_choices": 2,
            })

        except Exception as e:
            skipped += 1
            print(f"[warn] skipped one example in {dataset_name}-{split_name}: {type(e).__name__}: {e}")

    torch.save(rows, path)
    print(f"[cache] saved {path} ({len(rows)} examples, skipped={skipped})")
    return rows


# ============================================================
# DATASET FROM CACHED FEATURES
# ============================================================

class CachedMCQDataset(Dataset):
    def __init__(self, rows: List[Dict[str, Any]]):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        r = self.rows[idx]
        return {
            "q": r["q"],
            "qmask": r["qmask"],
            "choices": r["choices"],
            "cmask": r["cmask"],
            "choice_mask": r["choice_mask"],
            "label": torch.tensor(r["label"], dtype=torch.long),
            "num_choices": r["num_choices"],
            "idx": idx,
        }


def cached_collate(batch: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
    B = len(batch)
    T = batch[0]["q"].size(0)
    D = batch[0]["q"].size(1)
    K = batch[0]["choices"].size(0)
    V = batch[0]["choices"].size(1)

    q = torch.stack([item["q"] for item in batch], dim=0)
    qmask = torch.stack([item["qmask"] for item in batch], dim=0)

    choices = torch.zeros(B, K, V, T, D, dtype=torch.float32)
    cmask = torch.ones(B, K, V, T, dtype=torch.bool)
    choice_mask = torch.zeros(B, K, dtype=torch.bool)
    labels = torch.zeros(B, dtype=torch.long)
    idxs = torch.zeros(B, dtype=torch.long)

    for i, item in enumerate(batch):
        choices[i] = item["choices"]
        cmask[i] = item["cmask"]
        choice_mask[i] = item["choice_mask"]
        labels[i] = item["label"]
        idxs[i] = item["idx"]

    return {
        "q": q,
        "qmask": qmask,
        "choices": choices,
        "cmask": cmask,
        "choice_mask": choice_mask,
        "label": labels,
        "idx": idxs,
    }


# ============================================================
# MODEL BUILD / LOAD
# ============================================================

def build_hlcm_from_cfg(cfg: GRPOConfig) -> HyperbolicLCM:
    return HyperbolicLCM(
        in_dim=cfg.in_dim,
        model_dim=cfg.model_dim,
        num_heads=cfg.num_heads,
        num_layers=cfg.num_layers,
        ffn_mult=cfg.ffn_mult,
        dropout=cfg.dropout,
        manifold_c=cfg.manifold_c,
        causal=cfg.causal,
        input_scale=cfg.input_scale,
        input_max_norm=cfg.input_max_norm,
    )


def load_pretrained_hlcm(cfg: GRPOConfig, device: torch.device) -> HyperbolicLCM:
    model = build_hlcm_from_cfg(cfg).to(device)

    if not os.path.exists(cfg.ckpt_path):
        raise FileNotFoundError(f"Checkpoint not found: {cfg.ckpt_path}")

    obj = torch.load(cfg.ckpt_path, map_location="cpu")
    state = obj["model"] if "model" in obj else obj

    missing, unexpected = model.load_state_dict(state, strict=False)
    print(f"[load] loaded HLCM from {cfg.ckpt_path}")
    print(f"[load] missing keys: {len(missing)}")
    print(f"[load] unexpected keys: {len(unexpected)}")
    return model


# ============================================================
# ENCODING / LOGITS / LOSSES
# ============================================================

def hlcm_last_tangent(
    model: HyperbolicLCM,
    x: torch.Tensor,
    pad_mask: torch.Tensor,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
) -> torch.Tensor:
    device = next(model.parameters()).device
    x = x.to(device, non_blocking=True)
    pad_mask = pad_mask.to(device, non_blocking=True)

    if mu is not None and sigma is not None:
        x = (x - mu.view(1, 1, -1)) / sigma.view(1, 1, -1)

    h = model(x)
    h_tan = model.manifold.logmap0(h)

    B, T, D = h_tan.shape
    out = torch.empty((B, D), device=h_tan.device, dtype=h_tan.dtype)

    for i in range(B):
        valid = (~pad_mask[i]).nonzero(as_tuple=False).view(-1)
        j = int(valid[-1].item()) if valid.numel() else 0
        out[i] = h_tan[i, j]

    return out


def mcq_logits_hlcm(
    model: HyperbolicLCM,
    batch: Dict[str, torch.Tensor],
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    logit_temperature: float,
) -> torch.Tensor:
    device = next(model.parameters()).device

    q = batch["q"].to(device, non_blocking=True)                # [B, T, D]
    qmask = batch["qmask"].to(device, non_blocking=True)        # [B, T]
    choices = batch["choices"].to(device, non_blocking=True)    # [B, K, V, T, D]
    cmask = batch["cmask"].to(device, non_blocking=True)        # [B, K, V, T]
    choice_mask = batch["choice_mask"].to(device, non_blocking=True)

    B, K, V, T, D = choices.shape

    e_q = hlcm_last_tangent(model, q, qmask, mu=mu, sigma=sigma)
    e_q = F.normalize(e_q, dim=-1)

    logits_per_choice = []

    for k in range(K):
        ch = choices[:, k]
        ch_mask = cmask[:, k]

        flat = ch.reshape(B * V, T, D)
        flat_mask = ch_mask.reshape(B * V, T)

        e_c = hlcm_last_tangent(model, flat, flat_mask, mu=mu, sigma=sigma)
        e_c = e_c.reshape(B, V, -1)
        e_c = F.normalize(e_c, dim=-1)

        sims = torch.einsum("bd,bvd->bv", e_q, e_c)
        choice_logit = sims.mean(dim=1)
        logits_per_choice.append(choice_logit)

    logits = torch.stack(logits_per_choice, dim=1)
    logits = logits / max(logit_temperature, 1e-6)

    min_val = torch.finfo(logits.dtype).min
    logits = logits.masked_fill(~choice_mask, min_val)
    return logits


def compute_class_weights(rows: List[Dict[str, Any]]) -> torch.Tensor:
    counts = torch.zeros(2, dtype=torch.float32)
    for r in rows:
        counts[int(r["label"])] += 1.0
    counts = counts.clamp_min(1.0)
    weights = counts.sum() / (2.0 * counts)
    return weights


def mcq_loss_acc_hlcm(
    model: HyperbolicLCM,
    batch: Dict[str, torch.Tensor],
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    cfg: GRPOConfig,
    class_weights: Optional[torch.Tensor] = None,
):
    logits = mcq_logits_hlcm(
        model=model,
        batch=batch,
        mu=mu,
        sigma=sigma,
        logit_temperature=cfg.mcq_logit_temperature,
    )
    y = batch["label"].to(logits.device, non_blocking=True)

    weight = class_weights.to(logits.device) if class_weights is not None else None

    loss = F.cross_entropy(
        logits,
        y,
        weight=weight,
        label_smoothing=cfg.label_smoothing,
    )
    acc = (logits.argmax(dim=1) == y).float().mean()
    return loss, acc


def categorical_kl_from_logits(logits_p: torch.Tensor, logits_q: torch.Tensor) -> torch.Tensor:
    logp = F.log_softmax(logits_p, dim=-1)
    logq = F.log_softmax(logits_q, dim=-1)
    p = logp.exp()
    return torch.sum(p * (logp - logq), dim=-1)


def categorical_entropy_from_logits(logits: torch.Tensor) -> torch.Tensor:
    logp = F.log_softmax(logits, dim=-1)
    p = logp.exp()
    return -torch.sum(p * logp, dim=-1)


def group_relative_advantages(rewards: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    mean = rewards.mean(dim=1, keepdim=True)
    std = rewards.std(dim=1, keepdim=True, unbiased=False)
    return (rewards - mean) / (std + eps)


@torch.no_grad()
def sample_group_actions(
    logits: torch.Tensor,
    group_size: int,
    policy_temperature: float,
) -> torch.Tensor:
    scaled = logits / max(policy_temperature, 1e-6)
    dist = torch.distributions.Categorical(logits=scaled)
    actions = [dist.sample() for _ in range(group_size)]
    return torch.stack(actions, dim=1)


def rewards_from_actions(
    actions: torch.Tensor,
    labels: torch.Tensor,
    reward_correct: float,
    reward_incorrect: float,
) -> torch.Tensor:
    correct = (actions == labels.unsqueeze(1))
    return torch.where(
        correct,
        torch.full_like(actions, fill_value=reward_correct, dtype=torch.float32),
        torch.full_like(actions, fill_value=reward_incorrect, dtype=torch.float32),
    )


def grpo_loss_hlcm_cached_ref(
    model: HyperbolicLCM,
    batch: Dict[str, torch.Tensor],
    ref_logits: torch.Tensor,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    cfg: GRPOConfig,
):
    device = next(model.parameters()).device
    labels = batch["label"].to(device, non_blocking=True)
    ref_logits = ref_logits.to(device, non_blocking=True)

    logits = mcq_logits_hlcm(
        model=model,
        batch=batch,
        mu=mu,
        sigma=sigma,
        logit_temperature=cfg.mcq_logit_temperature,
    )

    with torch.no_grad():
        actions = sample_group_actions(
            logits=logits.detach(),
            group_size=cfg.grpo_group_size,
            policy_temperature=cfg.grpo_policy_temperature,
        )

        rewards = rewards_from_actions(
            actions=actions,
            labels=labels,
            reward_correct=cfg.reward_correct,
            reward_incorrect=cfg.reward_incorrect,
        )

        advantages = group_relative_advantages(rewards) if cfg.use_group_relative_advantage else rewards

    scaled_logits = logits / max(cfg.grpo_policy_temperature, 1e-6)
    log_probs = F.log_softmax(scaled_logits, dim=-1)
    sampled_logprobs = log_probs.gather(1, actions)

    policy_loss = -(advantages * sampled_logprobs).mean()

    ref_scaled_logits = ref_logits / max(cfg.grpo_policy_temperature, 1e-6)
    kl = categorical_kl_from_logits(scaled_logits, ref_scaled_logits)
    kl_loss = kl.mean()

    entropy = categorical_entropy_from_logits(scaled_logits).mean()
    total_loss = policy_loss + cfg.grpo_beta_kl * kl_loss - cfg.entropy_bonus * entropy

    with torch.no_grad():
        pred = logits.argmax(dim=1)
        acc = (pred == labels).float().mean()

    stats = {
        "loss": float(total_loss.item()),
        "policy_loss": float(policy_loss.item()),
        "kl_loss": float(kl_loss.item()),
        "entropy": float(entropy.item()),
        "acc": float(acc.item()),
        "reward_mean": float(rewards.mean().item()),
        "reward_std": float(rewards.std(unbiased=False).item()),
        "adv_mean": float(advantages.mean().item()),
        "adv_std": float(advantages.std(unbiased=False).item()),
    }
    return total_loss, stats


# ============================================================
# EVAL
# ============================================================

@torch.no_grad()
def evaluate_hlcm(
    model: HyperbolicLCM,
    loader: Optional[DataLoader],
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    cfg: GRPOConfig,
) -> Dict[str, float]:
    if loader is None:
        return {"loss": 0.0, "acc": 0.0}

    model.eval()
    tot_loss, tot_acc, n = 0.0, 0.0, 0

    for batch in loader:
        loss, acc = mcq_loss_acc_hlcm(
            model=model,
            batch=batch,
            mu=mu,
            sigma=sigma,
            cfg=cfg,
            class_weights=None,
        )
        bs = batch["label"].size(0)
        tot_loss += float(loss.item()) * bs
        tot_acc += float(acc.item()) * bs
        n += bs

    return {"loss": tot_loss / max(1, n), "acc": tot_acc / max(1, n)}


# ============================================================
# TRAINING HELPERS
# ============================================================

def make_optimizer_and_scheduler(
    trainable_params,
    lr: float,
    total_steps: int,
    warmup_ratio: float,
    weight_decay: float,
):
    trainable_params = list(trainable_params)
    if len(trainable_params) == 0:
        raise ValueError(
            "optimizer got an empty parameter list. "
            "No trainable parameters were found. "
            "Use finetune_mode='last_blocks' or 'full'."
        )

    opt = torch.optim.AdamW(trainable_params, lr=lr, weight_decay=weight_decay)
    warmup_steps = int(warmup_ratio * total_steps)

    def lr_lambda(step: int):
        if step < warmup_steps:
            return step / max(1, warmup_steps)
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1.0 + math.cos(math.pi * min(1.0, progress)))

    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
    return opt, sched


# ============================================================
# REFERENCE LOGIT CACHING
# ============================================================

@torch.no_grad()
def precompute_reference_logits(
    model: HyperbolicLCM,
    dataset: CachedMCQDataset,
    cfg: GRPOConfig,
    device: torch.device,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    out_path: str,
):
    if os.path.exists(out_path):
        print(f"[ref_logits] loading existing {out_path}")
        return torch.load(out_path)

    print(f"[ref_logits] building {out_path}")
    model.eval()

    rows = []
    loader = DataLoader(
        dataset,
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=cached_collate,
        drop_last=False,
    )

    for batch in tqdm(loader, desc="precompute_ref_logits"):
        logits = mcq_logits_hlcm(
            model=model,
            batch=batch,
            mu=mu,
            sigma=sigma,
            logit_temperature=cfg.mcq_logit_temperature,
        )

        logits_cpu = logits.detach().cpu().float()
        choice_mask_cpu = batch["choice_mask"].cpu()
        idx_cpu = batch["idx"].cpu()

        for i in range(logits_cpu.size(0)):
            valid_k = int(choice_mask_cpu[i].sum().item())
            rows.append({
                "idx": int(idx_cpu[i].item()),
                "ref_logits": logits_cpu[i, :valid_k].clone(),
            })

    rows = sorted(rows, key=lambda x: x["idx"])
    torch.save(rows, out_path)
    print(f"[ref_logits] saved {out_path} ({len(rows)} rows)")
    return rows


# ============================================================
# STAGE 1: SFT
# ============================================================

def run_stage_supervised_hlcm(
    model: HyperbolicLCM,
    train_loader: DataLoader,
    eval_loader: DataLoader,
    out_dir: str,
    metadata: dict,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    cfg: GRPOConfig,
    device: torch.device,
    class_weights: Optional[torch.Tensor] = None,
):
    trainable_params = [p for p in model.parameters() if p.requires_grad]

    steps_per_epoch = math.ceil(len(train_loader) / max(1, cfg.grad_accum_steps))
    total_steps = max(1, cfg.sft_epochs * steps_per_epoch)

    opt, sched = make_optimizer_and_scheduler(
        trainable_params=trainable_params,
        lr=cfg.sft_lr,
        total_steps=total_steps,
        warmup_ratio=cfg.sft_warmup_ratio,
        weight_decay=cfg.weight_decay,
    )

    use_amp = cfg.use_bf16 and device.type == "cuda"
    scaler = amp.GradScaler(
        "cuda",
        enabled=(device.type == "cuda" and use_amp and not torch.cuda.is_bf16_supported())
    )

    train_csv = os.path.join(out_dir, "sft_train_log.csv")
    eval_csv = os.path.join(out_dir, "sft_eval_log.csv")

    base = evaluate_hlcm(model, eval_loader, mu, sigma, cfg)
    print(f"[SFT][BASE] loss={base['loss']:.4f} acc={base['acc']:.4f}")
    cuda_cleanup()

    append_dict_to_csv(eval_csv, {
        "phase": "base",
        "epoch": 0,
        "train_loss": "",
        "train_acc": "",
        "eval_loss": base["loss"],
        "eval_acc": base["acc"],
    })

    best_acc = base["acc"]
    best_state = clone_state_dict_to_cpu(model)
    t_train_start = time.time()
    global_opt_step = 0

    for epoch in range(1, cfg.sft_epochs + 1):
        model.train()
        opt.zero_grad(set_to_none=True)

        epoch_loss_sum = 0.0
        epoch_acc_sum = 0.0
        epoch_count = 0

        pbar = tqdm(train_loader, desc=f"sft epoch {epoch}/{cfg.sft_epochs}", dynamic_ncols=True)

        for batch_idx, batch in enumerate(pbar, start=1):
            if device.type == "cuda":
                with amp.autocast(device_type="cuda", enabled=use_amp, dtype=build_amp_dtype(device)):
                    loss, acc = mcq_loss_acc_hlcm(
                        model, batch, mu, sigma, cfg, class_weights=class_weights
                    )
                    loss_for_backward = loss / cfg.grad_accum_steps
            else:
                loss, acc = mcq_loss_acc_hlcm(
                    model, batch, mu, sigma, cfg, class_weights=class_weights
                )
                loss_for_backward = loss / cfg.grad_accum_steps

            bs = batch["label"].size(0)
            epoch_loss_sum += float(loss.item()) * bs
            epoch_acc_sum += float(acc.item()) * bs
            epoch_count += bs

            if scaler.is_enabled():
                scaler.scale(loss_for_backward).backward()
            else:
                loss_for_backward.backward()

            do_step = (batch_idx % cfg.grad_accum_steps == 0) or (batch_idx == len(train_loader))
            if do_step:
                if scaler.is_enabled():
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(trainable_params, cfg.max_grad_norm)
                    scaler.step(opt)
                    scaler.update()
                else:
                    torch.nn.utils.clip_grad_norm_(trainable_params, cfg.max_grad_norm)
                    opt.step()

                sched.step()
                opt.zero_grad(set_to_none=True)
                global_opt_step += 1

            pbar.set_postfix(
                loss=f"{epoch_loss_sum / max(1, epoch_count):.4f}",
                acc=f"{epoch_acc_sum / max(1, epoch_count):.4f}",
                lr=f"{opt.param_groups[0]['lr']:.2e}",
            )

        train_loss = epoch_loss_sum / max(1, epoch_count)
        train_acc = epoch_acc_sum / max(1, epoch_count)

        ev = evaluate_hlcm(model, eval_loader, mu, sigma, cfg)
        cuda_cleanup()

        append_dict_to_csv(train_csv, {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "lr": opt.param_groups[0]["lr"],
            "global_opt_step": global_opt_step,
        })

        append_dict_to_csv(eval_csv, {
            "phase": "eval",
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "eval_loss": ev["loss"],
            "eval_acc": ev["acc"],
        })

        print(
            f"[SFT][epoch {epoch}/{cfg.sft_epochs}] "
            f"train_loss={train_loss:.4f} "
            f"train_acc={train_acc:.4f} "
            f"eval_loss={ev['loss']:.4f} "
            f"eval_acc={ev['acc']:.4f}"
        )

        if ev["acc"] > best_acc:
            best_acc = ev["acc"]
            best_state = clone_state_dict_to_cpu(model)
            save_checkpoint(
                {
                    "model": best_state,
                    "stage": "sft_best",
                    "best_eval_acc": best_acc,
                    "epoch": epoch,
                    "global_opt_step": global_opt_step,
                    **metadata,
                },
                os.path.join(out_dir, "sft_best.pt"),
            )
            print("  saved sft_best.pt")

        save_checkpoint(
            {
                "model": clone_state_dict_to_cpu(model),
                "stage": "sft_last",
                "epoch": epoch,
                "global_opt_step": global_opt_step,
                **metadata,
            },
            os.path.join(out_dir, "sft_last.pt"),
        )

    total_minutes = (time.time() - t_train_start) / 60.0
    model.load_state_dict(best_state, strict=True)

    del opt
    del sched
    del scaler
    cuda_cleanup()

    print(f"[SFT][FINAL] best_acc={best_acc:.4f} total_train_time={total_minutes:.2f} min")
    return {
        "best_acc": best_acc,
        "best_state": best_state,
        "total_minutes": total_minutes,
    }


# ============================================================
# STAGE 2: GRPO WITH CACHED REF LOGITS
# ============================================================

def run_stage_grpo_hlcm_cached_ref(
    model: HyperbolicLCM,
    train_loader: DataLoader,
    eval_loader: DataLoader,
    ref_logits_rows: List[Dict[str, Any]],
    out_dir: str,
    metadata: dict,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    cfg: GRPOConfig,
    device: torch.device,
):
    ref_logits_map = {int(r["idx"]): r["ref_logits"] for r in ref_logits_rows}

    trainable_params = [p for p in model.parameters() if p.requires_grad]

    steps_per_epoch = math.ceil(len(train_loader) / max(1, cfg.grad_accum_steps))
    total_steps = max(1, cfg.grpo_epochs * steps_per_epoch)

    opt, sched = make_optimizer_and_scheduler(
        trainable_params=trainable_params,
        lr=cfg.grpo_lr,
        total_steps=total_steps,
        warmup_ratio=cfg.grpo_warmup_ratio,
        weight_decay=cfg.weight_decay,
    )

    use_amp = cfg.use_bf16 and device.type == "cuda"
    scaler = amp.GradScaler(
        "cuda",
        enabled=(device.type == "cuda" and use_amp and not torch.cuda.is_bf16_supported())
    )

    train_csv = os.path.join(out_dir, "grpo_train_log.csv")
    eval_csv = os.path.join(out_dir, "grpo_eval_log.csv")

    base = evaluate_hlcm(model, eval_loader, mu, sigma, cfg)
    print(f"[GRPO][BASE] loss={base['loss']:.4f} acc={base['acc']:.4f}")
    cuda_cleanup()

    append_dict_to_csv(eval_csv, {
        "phase": "base",
        "epoch": 0,
        "train_loss": "",
        "train_acc": "",
        "eval_loss": base["loss"],
        "eval_acc": base["acc"],
    })

    best_acc = base["acc"]
    best_state = clone_state_dict_to_cpu(model)
    t_train_start = time.time()
    global_opt_step = 0

    for epoch in range(1, cfg.grpo_epochs + 1):
        model.train()
        opt.zero_grad(set_to_none=True)

        epoch_loss_sum = 0.0
        epoch_acc_sum = 0.0
        epoch_count = 0
        last_stats = None

        pbar = tqdm(train_loader, desc=f"grpo epoch {epoch}/{cfg.grpo_epochs}", dynamic_ncols=True)

        for batch_idx, batch in enumerate(pbar, start=1):
            idxs = batch["idx"].tolist()
            max_k = int(batch["choice_mask"].sum(dim=1).max().item())
            ref_logits_batch = torch.full((len(idxs), max_k), fill_value=-1e9, dtype=torch.float32)

            for i, ex_idx in enumerate(idxs):
                r = ref_logits_map[int(ex_idx)]
                k = r.numel()
                ref_logits_batch[i, :k] = r

            if device.type == "cuda":
                with amp.autocast(device_type="cuda", enabled=use_amp, dtype=build_amp_dtype(device)):
                    loss, stats = grpo_loss_hlcm_cached_ref(
                        model=model,
                        batch=batch,
                        ref_logits=ref_logits_batch,
                        mu=mu,
                        sigma=sigma,
                        cfg=cfg,
                    )
                    loss_for_backward = loss / cfg.grad_accum_steps
            else:
                loss, stats = grpo_loss_hlcm_cached_ref(
                    model=model,
                    batch=batch,
                    ref_logits=ref_logits_batch,
                    mu=mu,
                    sigma=sigma,
                    cfg=cfg,
                )
                loss_for_backward = loss / cfg.grad_accum_steps

            bs = batch["label"].size(0)
            epoch_loss_sum += float(stats["loss"]) * bs
            epoch_acc_sum += float(stats["acc"]) * bs
            epoch_count += bs
            last_stats = stats

            if scaler.is_enabled():
                scaler.scale(loss_for_backward).backward()
            else:
                loss_for_backward.backward()

            do_step = (batch_idx % cfg.grad_accum_steps == 0) or (batch_idx == len(train_loader))
            if do_step:
                if scaler.is_enabled():
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(trainable_params, cfg.max_grad_norm)
                    scaler.step(opt)
                    scaler.update()
                else:
                    torch.nn.utils.clip_grad_norm_(trainable_params, cfg.max_grad_norm)
                    opt.step()

                sched.step()
                opt.zero_grad(set_to_none=True)
                global_opt_step += 1

            pbar.set_postfix(
                loss=f"{epoch_loss_sum / max(1, epoch_count):.4f}",
                acc=f"{epoch_acc_sum / max(1, epoch_count):.4f}",
                policy=f"{last_stats['policy_loss']:.4f}" if last_stats else "0.0000",
                kl=f"{last_stats['kl_loss']:.4f}" if last_stats else "0.0000",
                lr=f"{opt.param_groups[0]['lr']:.2e}",
            )

        train_loss = epoch_loss_sum / max(1, epoch_count)
        train_acc = epoch_acc_sum / max(1, epoch_count)

        ev = evaluate_hlcm(model, eval_loader, mu, sigma, cfg)
        cuda_cleanup()

        append_dict_to_csv(train_csv, {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "policy_loss": None if last_stats is None else last_stats["policy_loss"],
            "kl_loss": None if last_stats is None else last_stats["kl_loss"],
            "entropy": None if last_stats is None else last_stats["entropy"],
            "reward_mean": None if last_stats is None else last_stats["reward_mean"],
            "reward_std": None if last_stats is None else last_stats["reward_std"],
            "adv_mean": None if last_stats is None else last_stats["adv_mean"],
            "adv_std": None if last_stats is None else last_stats["adv_std"],
            "lr": opt.param_groups[0]["lr"],
            "global_opt_step": global_opt_step,
        })

        append_dict_to_csv(eval_csv, {
            "phase": "eval",
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "eval_loss": ev["loss"],
            "eval_acc": ev["acc"],
        })

        print(
            f"[GRPO][epoch {epoch}/{cfg.grpo_epochs}] "
            f"train_loss={train_loss:.4f} "
            f"train_acc={train_acc:.4f} "
            f"eval_loss={ev['loss']:.4f} "
            f"eval_acc={ev['acc']:.4f}"
        )

        if ev["acc"] > best_acc:
            best_acc = ev["acc"]
            best_state = clone_state_dict_to_cpu(model)
            save_checkpoint(
                {
                    "model": best_state,
                    "stage": "grpo_best",
                    "best_eval_acc": best_acc,
                    "epoch": epoch,
                    "global_opt_step": global_opt_step,
                    **metadata,
                },
                os.path.join(out_dir, "grpo_best.pt"),
            )
            print("  saved grpo_best.pt")

        save_checkpoint(
            {
                "model": clone_state_dict_to_cpu(model),
                "stage": "grpo_last",
                "epoch": epoch,
                "global_opt_step": global_opt_step,
                **metadata,
            },
            os.path.join(out_dir, "grpo_last.pt"),
        )

    total_minutes = (time.time() - t_train_start) / 60.0
    model.load_state_dict(best_state, strict=True)

    del opt
    del sched
    del scaler
    cuda_cleanup()

    print(f"[GRPO][FINAL] best_acc={best_acc:.4f} total_train_time={total_minutes:.2f} min")
    return {
        "best_acc": best_acc,
        "best_state": best_state,
        "total_minutes": total_minutes,
    }


# ============================================================
# TRAIN ONE DATASET
# ============================================================

def train_boolq_hybrid_hlcm(dataset_name: str, cfg: GRPOConfig, device: torch.device):
    print(f"\n==================== {dataset_name} ====================")
    out_dir = os.path.join(cfg.out_dir, dataset_name.replace("/", "_"))
    ensure_dir(out_dir)
    ensure_dir(cfg.cache_dir)
    ensure_dir(cfg.ref_logits_dir)

    conceptizer_device = torch.device(cfg.conceptizer_device)
    conceptizer = DebertaConceptizer(
        model_name=cfg.encoder_name,
        chunk_tok_len=cfg.chunk_tok_len,
        seq_len=cfg.seq_len,
        batch_size=cfg.encoder_batch_size,
        device=conceptizer_device,
    )

    train_hf, eval_hf = load_boolq_local(cfg)

    train_rows = build_or_load_cached_split(cfg, dataset_name, "train", train_hf, conceptizer)
    eval_rows = build_or_load_cached_split(cfg, dataset_name, "validation", eval_hf, conceptizer)

    del conceptizer
    cuda_cleanup()

    train_ds = CachedMCQDataset(train_rows)
    eval_ds = CachedMCQDataset(eval_rows)

    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.train_batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=cached_collate,
        drop_last=False,
    )

    eval_loader = DataLoader(
        eval_ds,
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=cached_collate,
        drop_last=False,
    )

    hlcm = load_pretrained_hlcm(cfg, device)
    apply_finetune_mode_hlcm(hlcm, cfg.finetune_mode, cfg.n_last_blocks)
    hlcm.train()

    trainable_params = sum(p.numel() for p in hlcm.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in hlcm.parameters())
    print(f"[params] total={total_params:,} trainable={trainable_params:,}")

    if trainable_params == 0:
        raise ValueError(
            "No trainable parameters found after applying finetune mode. "
            "Use finetune_mode='last_blocks' or 'full'."
        )

    mu, sigma = load_normalizer(cfg.normalizer_path, device)
    class_weights = compute_class_weights(train_rows) if cfg.use_class_weights else None
    print("[class_weights]", None if class_weights is None else class_weights.tolist())

    metadata_base = {
        "dataset": dataset_name,
        "arch": {
            "in_dim": cfg.in_dim,
            "model_dim": cfg.model_dim,
            "num_heads": cfg.num_heads,
            "num_layers": cfg.num_layers,
            "ffn_mult": cfg.ffn_mult,
            "manifold_c": cfg.manifold_c,
        },
        "concept_model": cfg.encoder_name,
        "chunk_tok_len": cfg.chunk_tok_len,
        "seq_len": cfg.seq_len,
        "finetune_mode": cfg.finetune_mode,
        "n_last_blocks": cfg.n_last_blocks,
        "pretrained_ckpt": cfg.ckpt_path,
        "bs_train": cfg.train_batch_size,
        "bs_eval": cfg.eval_batch_size,
        "grad_accum_steps": cfg.grad_accum_steps,
        "use_bf16": cfg.use_bf16,
        "num_train_examples": len(train_ds),
        "num_eval_examples": len(eval_ds),
        "train_parquet": cfg.hf_train_file,
        "validation_parquet": cfg.hf_validation_file,
        "label_smoothing": cfg.label_smoothing,
        "use_class_weights": cfg.use_class_weights,
    }

    # -------------------------
    # SFT
    # -------------------------
    print(f"\n========== {dataset_name} :: STAGE 1 / SFT ==========")
    sft_meta = {
        **metadata_base,
        "stage_name": "sft",
        "stage_epochs": cfg.sft_epochs,
        "stage_lr": cfg.sft_lr,
        "stage_warmup_ratio": cfg.sft_warmup_ratio,
        "mcq_logit_temperature": cfg.mcq_logit_temperature,
    }

    sft_result = run_stage_supervised_hlcm(
        model=hlcm,
        train_loader=train_loader,
        eval_loader=eval_loader,
        out_dir=out_dir,
        metadata=sft_meta,
        mu=mu,
        sigma=sigma,
        cfg=cfg,
        device=device,
        class_weights=class_weights,
    )

    save_checkpoint(
        {
            "model": clone_state_dict_to_cpu(hlcm),
            "stage": "after_sft",
            "best_eval_acc": sft_result["best_acc"],
            **sft_meta,
        },
        os.path.join(out_dir, "after_sft.pt"),
    )

    hlcm.load_state_dict(sft_result["best_state"], strict=True)
    del sft_result["best_state"]
    cuda_cleanup()
    hlcm.eval()

    # -------------------------
    # REF LOGITS FOR GRPO
    # -------------------------
    ref_logits_path = ref_logits_file_path(cfg, dataset_name, "train")
    ref_logits_rows = precompute_reference_logits(
        model=hlcm,
        dataset=train_ds,
        cfg=cfg,
        device=device,
        mu=mu,
        sigma=sigma,
        out_path=ref_logits_path,
    )
    cuda_cleanup()

    # -------------------------
    # GRPO
    # -------------------------
    print(f"\n========== {dataset_name} :: STAGE 2 / GRPO ==========")
    grpo_meta = {
        **metadata_base,
        "stage_name": "grpo_cached_ref_logits",
        "stage_epochs": cfg.grpo_epochs,
        "stage_lr": cfg.grpo_lr,
        "stage_warmup_ratio": cfg.grpo_warmup_ratio,
        "grpo_group_size": cfg.grpo_group_size,
        "grpo_beta_kl": cfg.grpo_beta_kl,
        "grpo_policy_temperature": cfg.grpo_policy_temperature,
        "mcq_logit_temperature": cfg.mcq_logit_temperature,
        "reward_correct": cfg.reward_correct,
        "reward_incorrect": cfg.reward_incorrect,
        "use_group_relative_advantage": cfg.use_group_relative_advantage,
        "entropy_bonus": cfg.entropy_bonus,
        "ref_logits_path": ref_logits_path,
    }

    hlcm.train()
    grpo_result = run_stage_grpo_hlcm_cached_ref(
        model=hlcm,
        train_loader=train_loader,
        eval_loader=eval_loader,
        ref_logits_rows=ref_logits_rows,
        out_dir=out_dir,
        metadata=grpo_meta,
        mu=mu,
        sigma=sigma,
        cfg=cfg,
        device=device,
    )

    hlcm.load_state_dict(grpo_result["best_state"], strict=True)
    del grpo_result["best_state"]
    cuda_cleanup()

    final_eval = evaluate_hlcm(hlcm, eval_loader, mu, sigma, cfg)
    mem = gpu_mem_mb(device)

    save_checkpoint(
        {
            "model": clone_state_dict_to_cpu(hlcm),
            "stage": "final_hybrid",
            "sft_best_acc": sft_result["best_acc"],
            "grpo_best_acc": grpo_result["best_acc"],
            "final_eval_loss": final_eval["loss"],
            "final_eval_acc": final_eval["acc"],
            "sft_total_minutes": sft_result["total_minutes"],
            "grpo_total_minutes": grpo_result["total_minutes"],
            "max_gpu_alloc_mb": mem["max_alloc_mb"],
            **metadata_base,
        },
        os.path.join(out_dir, "final_hybrid.pt"),
    )

    final_summary = {
        "dataset": dataset_name,
        "sft_best_acc": sft_result["best_acc"],
        "grpo_best_acc": grpo_result["best_acc"],
        "final_eval_loss": final_eval["loss"],
        "final_eval_acc": final_eval["acc"],
        "sft_total_minutes": sft_result["total_minutes"],
        "grpo_total_minutes": grpo_result["total_minutes"],
        "max_gpu_alloc_mb": mem["max_alloc_mb"],
        "finetune_mode": cfg.finetune_mode,
        "n_last_blocks": cfg.n_last_blocks,
        "bs_train": cfg.train_batch_size,
        "bs_eval": cfg.eval_batch_size,
        "grad_accum_steps": cfg.grad_accum_steps,
        "sft_lr": cfg.sft_lr,
        "grpo_lr": cfg.grpo_lr,
        "sft_epochs": cfg.sft_epochs,
        "grpo_epochs": cfg.grpo_epochs,
        "mcq_logit_temperature": cfg.mcq_logit_temperature,
        "grpo_policy_temperature": cfg.grpo_policy_temperature,
        "grpo_group_size": cfg.grpo_group_size,
        "grpo_beta_kl": cfg.grpo_beta_kl,
        "entropy_bonus": cfg.entropy_bonus,
        "label_smoothing": cfg.label_smoothing,
        "use_class_weights": cfg.use_class_weights,
        "seq_len": cfg.seq_len,
        "ref_logits_path": ref_logits_path,
        "train_parquet": cfg.hf_train_file,
        "validation_parquet": cfg.hf_validation_file,
    }

    write_single_row_csv(os.path.join(out_dir, "final_summary.csv"), final_summary)

    with open(os.path.join(out_dir, "final_summary.json"), "w") as f:
        json.dump(final_summary, f, indent=2)

    print(
        f"[FINAL] dataset={dataset_name} "
        f"sft_best_acc={sft_result['best_acc']:.4f} "
        f"grpo_best_acc={grpo_result['best_acc']:.4f} "
        f"final_eval_acc={final_eval['acc']:.4f} "
        f"max_gpu_alloc={mem['max_alloc_mb']:.1f} MB saved -> {out_dir}"
    )

    del hlcm
    del ref_logits_rows
    del sft_result
    del grpo_result
    cuda_cleanup()


# ============================================================
# MAIN
# ============================================================

def main():
    cfg = GRPOConfig()
    ensure_dir(cfg.out_dir)
    ensure_dir(cfg.cache_dir)
    ensure_dir(cfg.ref_logits_dir)

    set_seed(cfg.seed)
    device = pick_device(cfg.prefer_gpu_index)

    if device.type == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.set_float32_matmul_precision("high")

    print("Device:", device)
    print("Finetune mode:", cfg.finetune_mode)
    print(
        f"SFT epochs={cfg.sft_epochs}, GRPO epochs={cfg.grpo_epochs}, "
        f"bs_train={cfg.train_batch_size}, bs_eval={cfg.eval_batch_size}, grad_accum={cfg.grad_accum_steps}"
    )
    print(
        f"SFT lr={cfg.sft_lr}, GRPO lr={cfg.grpo_lr}, "
        f"temp={cfg.mcq_logit_temperature}, grpo_temp={cfg.grpo_policy_temperature}"
    )
    print(
        f"GRPO group_size={cfg.grpo_group_size}, beta_kl={cfg.grpo_beta_kl}, "
        f"entropy_bonus={cfg.entropy_bonus}"
    )
    print(
        f"label_smoothing={cfg.label_smoothing}, use_class_weights={cfg.use_class_weights}, "
        f"seq_len={cfg.seq_len}, n_last_blocks={cfg.n_last_blocks}"
    )

    all_t0 = time.time()

    for ds_name in cfg.datasets_to_run:
        train_boolq_hybrid_hlcm(ds_name, cfg, device)

    total_all = (time.time() - all_t0) / 60.0
    print("\nAll done. Outputs in:", cfg.out_dir)
    print(f"Total wall time: {total_all:.2f} min")


if __name__ == "__main__":
    main()

Device: cuda:0
Finetune mode: last_blocks
SFT epochs=2, GRPO epochs=1, bs_train=2, bs_eval=4, grad_accum=8
SFT lr=3e-05, GRPO lr=8e-06, temp=0.2, grpo_temp=0.8
GRPO group_size=4, beta_kl=0.03, entropy_bonus=0.002
label_smoothing=0.02, use_class_weights=True, seq_len=12, n_last_blocks=2

==================== boolq_local ====================


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[cache] building boolq_local / train


cache:boolq_local:train: 100%|██████████████████████████████████| 9427/9427 [49:19<00:00,  3.18it/s]


...

[SFT][epoch 2/2] train_loss=0.6933 train_acc=0.5331 eval_loss=0.6877 eval_acc=0.6110
  saved sft_best.pt
[SFT][FINAL] best_acc=0.6110 total_train_time=144.32 min
[ref_logits] building boolq_local_cached_ref_logits_grpo_better/boolq_local_train_ref_logits_tok256_seq12.pt


precompute_ref_logits: 100%|████████████████████████████████████| 2357/2357 [07:34<00:00,  5.19it/s]


[ref_logits] saved boolq_local_cached_ref_logits_grpo_better/boolq_local_train_ref_logits_tok256_seq12.pt (9427 rows)

========== boolq_local :: STAGE 2 / GRPO ==========
[GRPO][BASE] loss=0.6877 acc=0.6110


grpo epoch 1/1: 100%|█| 4714/4714 [1:08:24<00:00,  1.15it/s, acc=0.5442, kl=0.0005, loss=-0.0056, lr


[GRPO][epoch 1/1] train_loss=-0.0056 train_acc=0.5442 eval_loss=0.6813 eval_acc=0.6128
  saved grpo_best.pt
[GRPO][FINAL] best_acc=0.6128 total_train_time=71.76 min
[FINAL] dataset=boolq_local sft_best_acc=0.6110 grpo_best_acc=0.6128 final_eval_acc=0.6128 max_gpu_alloc=30761.1 MB saved -> runs/hlcm_boolq_local_grpo_better/boolq_local

All done. Outputs in: runs/hlcm_boolq_local_grpo_better
Total wall time: 305.26 min


In [1]:
# Accuracy + Precision/Recall/F1 + Precision@k/Recall@k/MRR

import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:128,expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import json, random, gc
from dataclasses import dataclass
from typing import List, Dict, Any, Tuple, Optional

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from torch import amp

from h_lcm1_Copy1 import HyperbolicLCM


@dataclass
class EvalConfig:
    dataset_name: str = "boolq_local"

    hf_validation_file: str = "/home/user/twovolume/Nisha/Finetune/boolq/validation-00000-of-00001.parquet"

    out_dir: str = "runs/hlcm_boolq_local_grpo_better"
    cache_dir: str = "boolq_local_cached_features_grpo_better"

    base_ckpt_path: str = "runs/hyperbolic_cluster/checkpoints/ckpt_best.pt"
    eval_ckpt_path: str = "runs/hlcm_boolq_local_grpo_better/boolq_local/grpo_best.pt"
    normalizer_path: str = "normalizer.pt"

    encoder_name: str = "microsoft/deberta-v3-small"
    chunk_tok_len: int = 256
    seq_len: int = 12
    encoder_batch_size: int = 16
    conceptizer_device: str = "cpu"

    in_dim: int = 768
    model_dim: int = 4096
    num_heads: int = 32
    num_layers: int = 12
    ffn_mult: int = 4
    dropout: float = 0.30
    manifold_c: float = 0.002
    causal: bool = True
    input_scale: float = 0.05
    input_max_norm: float = 1.0

    eval_batch_size: int = 4
    num_workers: int = 0
    mcq_logit_temperature: float = 0.2
    seed: int = 42
    prefer_gpu_index: int = 0


BOOLQ_YES_TEMPLATES = [
    "Answer: yes",
    "The correct answer is yes.",
    "Based on the passage, the answer is yes.",
    "The statement is supported by the passage.",
]

BOOLQ_NO_TEMPLATES = [
    "Answer: no",
    "The correct answer is no.",
    "Based on the passage, the answer is no.",
    "The statement is not supported by the passage.",
]


def set_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def ensure_dir(path):
    os.makedirs(path, exist_ok=True)


def pick_device(prefer_gpu_index=0):
    if not torch.cuda.is_available():
        return torch.device("cpu")
    n = torch.cuda.device_count()
    idx = 0 if n == 1 else max(0, min(prefer_gpu_index, n - 1))
    torch.cuda.set_device(idx)
    return torch.device(f"cuda:{idx}")


def build_amp_dtype(device):
    if device.type != "cuda":
        return torch.float32
    return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16


def cuda_cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def load_normalizer(path, device):
    if not path or not os.path.exists(path):
        return None, None
    obj = torch.load(path, map_location="cpu")
    mu = obj["mu"].float().to(device)
    sigma = obj["sigma"].float().clamp_min(1e-4).to(device)
    return mu, sigma


class DebertaConceptizer:
    def __init__(self, model_name, chunk_tok_len, seq_len, batch_size, device):
        self.chunk_tok_len = int(chunk_tok_len)
        self.seq_len = int(seq_len)
        self.batch_size = int(batch_size)
        self.device = device

        self.tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        self.enc = AutoModel.from_pretrained(model_name).to(device)
        self.enc.eval()

        for p in self.enc.parameters():
            p.requires_grad = False

        self.hidden_size = int(self.enc.config.hidden_size)

    @torch.inference_mode()
    def _embed_chunk_texts(self, chunk_texts):
        if len(chunk_texts) == 0:
            return torch.empty(0, self.hidden_size, dtype=torch.float32)

        inputs = self.tok(
            chunk_texts,
            padding=True,
            truncation=True,
            max_length=self.chunk_tok_len,
            return_tensors="pt",
        )
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        if self.device.type == "cuda":
            with amp.autocast(device_type="cuda", enabled=True, dtype=build_amp_dtype(self.device)):
                out = self.enc(**inputs).last_hidden_state[:, 0, :]
        else:
            out = self.enc(**inputs).last_hidden_state[:, 0, :]

        return out.detach().cpu().float()

    def _pack_into_chunk_texts(self, text):
        ids = self.tok(text, add_special_tokens=False)["input_ids"]
        out = []
        for i in range(0, len(ids), self.chunk_tok_len):
            out.append(self.tok.decode(ids[i:i + self.chunk_tok_len], clean_up_tokenization_spaces=True))
            if len(out) >= self.seq_len:
                break
        return out[:self.seq_len]

    @torch.inference_mode()
    def encode_text_fixed(self, text):
        chunk_texts = self._pack_into_chunk_texts(text)
        seq = torch.zeros(self.seq_len, self.hidden_size, dtype=torch.float32)
        pad = torch.ones(self.seq_len, dtype=torch.bool)

        if not chunk_texts:
            return seq, pad

        vecs = []
        for i in range(0, len(chunk_texts), self.batch_size):
            vecs.append(self._embed_chunk_texts(chunk_texts[i:i + self.batch_size]))

        vecs = torch.cat(vecs, dim=0)
        c = min(vecs.size(0), self.seq_len)
        seq[:c] = vecs[:c]
        pad[:c] = False
        return seq, pad


def normalize_boolq_example(ex):
    passage = str(ex.get("passage", "")).strip()
    question = str(ex.get("question", "")).strip()
    answer = bool(ex.get("answer", False))

    if question and not question.endswith("?"):
        question += "?"

    q_text = (
        "Task: Answer the yes/no question using the passage.\n"
        f"Passage: {passage}\n"
        f"Question: {question}"
    )

    label = 1 if answer else 0
    return q_text, label, passage, question


def cache_file_path(cfg, split_name):
    return os.path.join(
        cfg.cache_dir,
        f"{cfg.dataset_name}_{split_name}_tok{cfg.chunk_tok_len}_seq{cfg.seq_len}.pt"
    )


def build_or_load_cached_validation(cfg, conceptizer):
    path = cache_file_path(cfg, "validation")

    if os.path.exists(path):
        print(f"[cache] loading {path}")
        return torch.load(path)

    raw = load_dataset("parquet", data_files={"validation": cfg.hf_validation_file})
    val_split = raw["validation"]

    rows = []
    skipped = 0

    for ex in tqdm(val_split, desc="building validation cache"):
        try:
            q_text, label, passage, question = normalize_boolq_example(ex)
            q_seq, q_pad = conceptizer.encode_text_fixed(q_text)

            no_variants = [f"Passage: {passage}\nQuestion: {question}\n{t}" for t in BOOLQ_NO_TEMPLATES]
            yes_variants = [f"Passage: {passage}\nQuestion: {question}\n{t}" for t in BOOLQ_YES_TEMPLATES]

            choice_groups, mask_groups = [], []

            for variants in [no_variants, yes_variants]:
                seqs, pads = [], []
                for txt in variants:
                    s, p = conceptizer.encode_text_fixed(txt)
                    seqs.append(s)
                    pads.append(p)
                choice_groups.append(torch.stack(seqs, dim=0))
                mask_groups.append(torch.stack(pads, dim=0))

            rows.append({
                "q": q_seq,
                "qmask": q_pad,
                "choices": torch.stack(choice_groups, dim=0),
                "cmask": torch.stack(mask_groups, dim=0),
                "choice_mask": torch.tensor([True, True], dtype=torch.bool),
                "label": int(label),
            })

        except Exception as e:
            skipped += 1
            print(f"[warn] skipped example: {type(e).__name__}: {e}")

    torch.save(rows, path)
    print(f"[cache] saved {path}, rows={len(rows)}, skipped={skipped}")
    return rows


class CachedMCQDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        r = self.rows[idx]
        return {
            "q": r["q"],
            "qmask": r["qmask"],
            "choices": r["choices"],
            "cmask": r["cmask"],
            "choice_mask": r["choice_mask"],
            "label": torch.tensor(r["label"], dtype=torch.long),
            "idx": idx,
        }


def cached_collate(batch):
    B = len(batch)
    T = batch[0]["q"].size(0)
    D = batch[0]["q"].size(1)
    K = batch[0]["choices"].size(0)
    V = batch[0]["choices"].size(1)

    q = torch.stack([x["q"] for x in batch], dim=0)
    qmask = torch.stack([x["qmask"] for x in batch], dim=0)

    choices = torch.zeros(B, K, V, T, D)
    cmask = torch.ones(B, K, V, T, dtype=torch.bool)
    choice_mask = torch.zeros(B, K, dtype=torch.bool)
    labels = torch.zeros(B, dtype=torch.long)

    for i, x in enumerate(batch):
        choices[i] = x["choices"]
        cmask[i] = x["cmask"]
        choice_mask[i] = x["choice_mask"]
        labels[i] = x["label"]

    return {
        "q": q,
        "qmask": qmask,
        "choices": choices,
        "cmask": cmask,
        "choice_mask": choice_mask,
        "label": labels,
    }


def build_hlcm_from_cfg(cfg):
    return HyperbolicLCM(
        in_dim=cfg.in_dim,
        model_dim=cfg.model_dim,
        num_heads=cfg.num_heads,
        num_layers=cfg.num_layers,
        ffn_mult=cfg.ffn_mult,
        dropout=cfg.dropout,
        manifold_c=cfg.manifold_c,
        causal=cfg.causal,
        input_scale=cfg.input_scale,
        input_max_norm=cfg.input_max_norm,
    )


def load_model(cfg, device):
    model = build_hlcm_from_cfg(cfg).to(device)

    base = torch.load(cfg.base_ckpt_path, map_location="cpu")
    base_state = base["model"] if isinstance(base, dict) and "model" in base else base
    model.load_state_dict(base_state, strict=False)

    ckpt = torch.load(cfg.eval_ckpt_path, map_location="cpu")
    state = ckpt["model"] if isinstance(ckpt, dict) and "model" in ckpt else ckpt
    missing, unexpected = model.load_state_dict(state, strict=False)

    print(f"[load] grpo_best: {cfg.eval_ckpt_path}")
    print(f"[load] missing={len(missing)}, unexpected={len(unexpected)}")

    model.eval()
    for p in model.parameters():
        p.requires_grad = False

    return model, ckpt if isinstance(ckpt, dict) else {}


def hlcm_last_tangent(model, x, pad_mask, mu, sigma):
    device = next(model.parameters()).device
    x = x.to(device)
    pad_mask = pad_mask.to(device)

    if mu is not None and sigma is not None:
        x = (x - mu.view(1, 1, -1)) / sigma.view(1, 1, -1)

    h = model(x)
    h_tan = model.manifold.logmap0(h)

    out = torch.empty((h_tan.size(0), h_tan.size(2)), device=h_tan.device, dtype=h_tan.dtype)

    for i in range(h_tan.size(0)):
        valid = (~pad_mask[i]).nonzero(as_tuple=False).view(-1)
        j = int(valid[-1].item()) if valid.numel() else 0
        out[i] = h_tan[i, j]

    return out


def mcq_logits_hlcm(model, batch, mu, sigma, temperature):
    device = next(model.parameters()).device

    q = batch["q"].to(device)
    qmask = batch["qmask"].to(device)
    choices = batch["choices"].to(device)
    cmask = batch["cmask"].to(device)
    choice_mask = batch["choice_mask"].to(device)

    B, K, V, T, D = choices.shape

    e_q = hlcm_last_tangent(model, q, qmask, mu, sigma)
    e_q = F.normalize(e_q, dim=-1)

    logits_per_choice = []

    for k in range(K):
        flat = choices[:, k].reshape(B * V, T, D)
        flat_mask = cmask[:, k].reshape(B * V, T)

        e_c = hlcm_last_tangent(model, flat, flat_mask, mu, sigma)
        e_c = e_c.reshape(B, V, -1)
        e_c = F.normalize(e_c, dim=-1)

        sims = torch.einsum("bd,bvd->bv", e_q, e_c)
        logits_per_choice.append(sims.mean(dim=1))

    logits = torch.stack(logits_per_choice, dim=1)
    logits = logits / max(temperature, 1e-6)
    logits = logits.masked_fill(~choice_mask, torch.finfo(logits.dtype).min)

    return logits


def safe_div(a, b):
    return a / b if b else 0.0


def classification_metrics(y_true, y_pred):
    out = {}

    for cls, name in [(0, "no"), (1, "yes")]:
        tp = sum(t == cls and p == cls for t, p in zip(y_true, y_pred))
        fp = sum(t != cls and p == cls for t, p in zip(y_true, y_pred))
        fn = sum(t == cls and p != cls for t, p in zip(y_true, y_pred))
        precision = safe_div(tp, tp + fp)
        recall = safe_div(tp, tp + fn)
        f1 = safe_div(2 * precision * recall, precision + recall)

        out[f"precision_{name}"] = precision
        out[f"recall_{name}"] = recall
        out[f"f1_{name}"] = f1
        out[f"support_{name}"] = sum(t == cls for t in y_true)

    out["macro_precision"] = (out["precision_no"] + out["precision_yes"]) / 2
    out["macro_recall"] = (out["recall_no"] + out["recall_yes"]) / 2
    out["macro_f1"] = (out["f1_no"] + out["f1_yes"]) / 2

    return out

def multiclass_brier_score(probs, labels, choice_mask):
    one_hot = torch.zeros_like(probs)
    one_hot.scatter_(1, labels.view(-1, 1), 1.0)

    sq_error = (probs - one_hot) ** 2
    sq_error = sq_error.masked_fill(~choice_mask, 0.0)

    return sq_error.sum(dim=1)


def expected_calibration_error(confidences, correctness, n_bins=15):
    ece = 0.0
    mce = 0.0

    if confidences.numel() == 0:
        return 0.0, 0.0

    for i in range(n_bins):
        lo = i / n_bins
        hi = (i + 1) / n_bins

        if i == n_bins - 1:
            mask = (confidences >= lo) & (confidences <= hi)
        else:
            mask = (confidences >= lo) & (confidences < hi)

        if mask.any():
            bin_conf = confidences[mask].mean()
            bin_acc = correctness[mask].float().mean()
            gap = torch.abs(bin_conf - bin_acc).item()
            weight = mask.float().mean().item()

            ece += weight * gap
            mce = max(mce, gap)

    return float(ece), float(mce)
    
@torch.no_grad()
def evaluate(model, loader, cfg, mu, sigma):
    total = 0
    total_loss = 0.0
    correct = 0
    total_brier = 0.0

    y_true, y_pred = [], []

    all_confidences = []
    all_correctness = []

    p1 = r1 = p2 = r2 = mrr = 0.0

    for batch in tqdm(loader, desc="eval"):
        logits = mcq_logits_hlcm(model, batch, mu, sigma, cfg.mcq_logit_temperature)
        labels = batch["label"].to(logits.device)
        choice_mask = batch["choice_mask"].to(logits.device)

        loss = F.cross_entropy(logits, labels)
        probs = F.softmax(logits, dim=-1)

        bs = labels.size(0)
        total += bs
        total_loss += float(loss.item()) * bs

        preds = logits.argmax(dim=-1)
        batch_correct = preds == labels
        correct += int(batch_correct.sum().item())

        brier_per_example = multiclass_brier_score(
            probs=probs,
            labels=labels,
            choice_mask=choice_mask,
        )
        total_brier += float(brier_per_example.sum().item())

        confidences = probs.max(dim=-1).values
        all_confidences.append(confidences.detach().cpu())
        all_correctness.append(batch_correct.detach().cpu().float())

        y_true.extend(labels.cpu().tolist())
        y_pred.extend(preds.cpu().tolist())

        ranked = torch.argsort(logits, dim=-1, descending=True)

        for i in range(bs):
            gold = int(labels[i].item())
            rank = int((ranked[i] == gold).nonzero(as_tuple=False).item()) + 1

            mrr += 1.0 / rank

            hit1 = 1.0 if rank <= 1 else 0.0
            hit2 = 1.0 if rank <= 2 else 0.0

            r1 += hit1
            p1 += hit1

            r2 += hit2
            p2 += hit2 / 2.0

    confidences_all = torch.cat(all_confidences, dim=0) if all_confidences else torch.empty(0)
    correctness_all = torch.cat(all_correctness, dim=0) if all_correctness else torch.empty(0)

    ece, mce = expected_calibration_error(
        confidences=confidences_all,
        correctness=correctness_all,
        n_bins=15,
    )

    metrics = {
        "loss": total_loss / max(total, 1),
        "nll": total_loss / max(total, 1),
        "accuracy": correct / max(total, 1),

        "brier_score": total_brier / max(total, 1),
        "ece": ece,
        "mce": mce,
        "ece_bins": 15,

        "precision@1": p1 / max(total, 1),
        "recall@1": r1 / max(total, 1),
        "precision@2": p2 / max(total, 1),
        "recall@2": r2 / max(total, 1),
        "mrr": mrr / max(total, 1),
        "num_examples": total,
    }

    metrics.update(classification_metrics(y_true, y_pred))
    return metrics

def main():
    cfg = EvalConfig()
    set_seed(cfg.seed)

    ensure_dir(cfg.out_dir)
    ensure_dir(cfg.cache_dir)

    device = pick_device(cfg.prefer_gpu_index)
    print("Device:", device)
    print("EVAL ONLY. No training.")

    conceptizer = DebertaConceptizer(
        cfg.encoder_name,
        cfg.chunk_tok_len,
        cfg.seq_len,
        cfg.encoder_batch_size,
        torch.device(cfg.conceptizer_device),
    )

    val_rows = build_or_load_cached_validation(cfg, conceptizer)
    del conceptizer
    cuda_cleanup()

    val_loader = DataLoader(
        CachedMCQDataset(val_rows),
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=cached_collate,
    )

    model, ckpt_info = load_model(cfg, device)
    mu, sigma = load_normalizer(cfg.normalizer_path, device)

    metrics = evaluate(model, val_loader, cfg, mu, sigma)

    result = {
        "dataset": cfg.dataset_name,
        "split": "validation",
        "checkpoint": cfg.eval_ckpt_path,
        "checkpoint_stage": ckpt_info.get("stage", None),
        "checkpoint_best_eval_acc": ckpt_info.get("best_eval_acc", None),
        "metrics": metrics,
        "labels": {
            "0": "no",
            "1": "yes"
        }
    }

    save_path = os.path.join(cfg.out_dir, cfg.dataset_name, "grpo_best_precision_recall_eval.json")
    ensure_dir(os.path.dirname(save_path))

    with open(save_path, "w") as f:
        json.dump(result, f, indent=2)

    print(json.dumps(result, indent=2))
    print(f"Saved to: {save_path}")


if __name__ == "__main__":
    main()

Device: cuda:0
EVAL ONLY. No training.


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[cache] loading boolq_local_cached_features_grpo_better/boolq_local_validation_tok256_seq12.pt
[load] grpo_best: runs/hlcm_boolq_local_grpo_better/boolq_local/grpo_best.pt
[load] missing=0, unexpected=0


eval: 100%|███████████████████████████████████████████████████████| 818/818 [03:40<00:00,  3.71it/s]


{
  "dataset": "boolq_local",
  "split": "validation",
  "checkpoint": "runs/hlcm_boolq_local_grpo_better/boolq_local/grpo_best.pt",
  "checkpoint_stage": "grpo_best",
  "checkpoint_best_eval_acc": 0.6128440366972477,
  "metrics": {
    "loss": 0.6810218236497418,
    "nll": 0.6810218236497418,
    "accuracy": 0.6128440366972477,
    "brier_score": 0.4878924102411358,
    "ece": 0.0859635814264923,
    "mce": 0.12913018465042114,
    "ece_bins": 15,
    "precision@1": 0.6128440366972477,
    "recall@1": 0.6128440366972477,
    "precision@2": 0.5,
    "recall@2": 1.0,
    "mrr": 0.8064220183486238,
    "num_examples": 3270,
    "precision_no": 0.2698412698412698,
    "recall_no": 0.0137429264349232,
    "f1_no": 0.026153846153846153,
    "support_no": 1237,
    "precision_yes": 0.6195821640162146,
    "recall_yes": 0.9773733398917855,
    "f1_yes": 0.7583969465648855,
    "support_yes": 2033,
    "macro_precision": 0.44471171692874223,
    "macro_recall": 0.4955581331633544,
    "macro_

In [1]:
#inference
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:128,expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import time
import json
import random
import gc
from dataclasses import dataclass
from typing import List, Dict, Any, Tuple, Optional

import torch
import torch.nn.functional as F
from torch import amp
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel

from h_lcm1_Copy1 import HyperbolicLCM


# ============================================================
# CONFIG
# ============================================================

@dataclass
class InferenceConfig:
    dataset_name: str = "boolq_local"

    hf_train_file: str = "/home/user/twovolume/Nisha/Finetune/boolq/train-00000-of-00001.parquet"
    hf_validation_file: str = "/home/user/twovolume/Nisha/Finetune/boolq/validation-00000-of-00001.parquet"

    out_dir: str = "runs/hlcm_boolq_local_grpo_better"
    cache_dir: str = "boolq_local_cached_features_grpo_better"

    grpo_ckpt_path: str = "runs/hlcm_boolq_local_grpo_better/boolq_local/grpo_best.pt"
    normalizer_path: str = "normalizer.pt"

    encoder_name: str = "microsoft/deberta-v3-small"
    chunk_tok_len: int = 256
    seq_len: int = 12
    encoder_batch_size: int = 16
    conceptizer_device: str = "cpu"

    in_dim: int = 768
    model_dim: int = 4096
    num_heads: int = 32
    num_layers: int = 12
    ffn_mult: int = 4
    dropout: float = 0.30
    manifold_c: float = 0.002
    causal: bool = True
    input_scale: float = 0.05
    input_max_norm: float = 1.0

    eval_batch_size: int = 4
    num_workers: int = 0

    mcq_logit_temperature: float = 0.2
    min_valid_choices: int = 2

    seed: int = 42
    prefer_gpu_index: int = 0

    split: str = "validation"
    build_cache_if_missing: bool = True


BOOLQ_YES_TEMPLATES = [
    "Answer: yes",
    "The correct answer is yes.",
    "Based on the passage, the answer is yes.",
    "The statement is supported by the passage.",
]

BOOLQ_NO_TEMPLATES = [
    "Answer: no",
    "The correct answer is no.",
    "Based on the passage, the answer is no.",
    "The statement is not supported by the passage.",
]


# ============================================================
# UTILS
# ============================================================

def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


def fmt_hms(seconds: float) -> str:
    seconds = int(max(0, seconds))
    return f"{seconds // 3600:02d}:{(seconds % 3600) // 60:02d}:{seconds % 60:02d}"


def pick_device(prefer_gpu_index: int = 0) -> torch.device:
    if not torch.cuda.is_available():
        return torch.device("cpu")

    n = torch.cuda.device_count()
    idx = 0 if n == 1 else max(0, min(prefer_gpu_index, n - 1))
    torch.cuda.set_device(idx)
    return torch.device(f"cuda:{idx}")


def build_amp_dtype(device: torch.device) -> torch.dtype:
    if device.type != "cuda":
        return torch.float32
    return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16


def cuda_cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def load_normalizer(normalizer_path: str, device: torch.device):
    if not normalizer_path or not os.path.exists(normalizer_path):
        print("[normalizer] not found; using None")
        return None, None

    obj = torch.load(normalizer_path, map_location="cpu")
    mu = obj["mu"].float().to(device)
    sigma = obj["sigma"].float().clamp_min(1e-4).to(device)

    print(f"[normalizer] loaded from {normalizer_path}")
    return mu, sigma


# ============================================================
# CONCEPTIZER
# ============================================================

class DebertaConceptizer:
    def __init__(
        self,
        model_name: str,
        chunk_tok_len: int,
        seq_len: int,
        batch_size: int,
        device: torch.device,
    ):
        self.chunk_tok_len = int(chunk_tok_len)
        self.seq_len = int(seq_len)
        self.batch_size = int(batch_size)
        self.device = device

        self.tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        self.enc = AutoModel.from_pretrained(model_name).to(device)
        self.enc.eval()

        for p in self.enc.parameters():
            p.requires_grad = False

        self.hidden_size = int(self.enc.config.hidden_size)

    @torch.inference_mode()
    def _embed_chunk_texts(self, chunk_texts: List[str]) -> torch.Tensor:
        if len(chunk_texts) == 0:
            return torch.empty(0, self.hidden_size, dtype=torch.float32)

        inputs = self.tok(
            chunk_texts,
            padding=True,
            truncation=True,
            max_length=self.chunk_tok_len,
            return_tensors="pt",
        )
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        if self.device.type == "cuda":
            with amp.autocast(
                device_type="cuda",
                enabled=True,
                dtype=build_amp_dtype(self.device),
            ):
                out = self.enc(**inputs).last_hidden_state[:, 0, :]
        else:
            out = self.enc(**inputs).last_hidden_state[:, 0, :]

        return out.detach().cpu().float()

    def _pack_into_chunk_texts(self, text: str) -> List[str]:
        ids = self.tok(text, add_special_tokens=False)["input_ids"]

        if not ids:
            return []

        chunks = []
        for i in range(0, len(ids), self.chunk_tok_len):
            chunks.append(
                self.tok.decode(
                    ids[i:i + self.chunk_tok_len],
                    clean_up_tokenization_spaces=True,
                )
            )
            if len(chunks) >= self.seq_len:
                break

        return chunks[:self.seq_len]

    @torch.inference_mode()
    def encode_text_fixed(self, text: str) -> Tuple[torch.Tensor, torch.Tensor]:
        chunk_texts = self._pack_into_chunk_texts(text)

        seq = torch.zeros(self.seq_len, self.hidden_size, dtype=torch.float32)
        pad = torch.ones(self.seq_len, dtype=torch.bool)

        if not chunk_texts:
            return seq, pad

        vecs = []
        for i in range(0, len(chunk_texts), self.batch_size):
            vecs.append(self._embed_chunk_texts(chunk_texts[i:i + self.batch_size]))

        vecs = torch.cat(vecs, dim=0)
        c = min(vecs.size(0), self.seq_len)

        seq[:c] = vecs[:c]
        pad[:c] = False

        return seq, pad


# ============================================================
# BOOLQ DATA
# ============================================================

def load_boolq_local(cfg: InferenceConfig):
    if not os.path.exists(cfg.hf_train_file):
        raise FileNotFoundError(f"Train parquet not found: {cfg.hf_train_file}")
    if not os.path.exists(cfg.hf_validation_file):
        raise FileNotFoundError(f"Validation parquet not found: {cfg.hf_validation_file}")

    raw = load_dataset(
        "parquet",
        data_files={
            "train": cfg.hf_train_file,
            "validation": cfg.hf_validation_file,
        },
    )

    return raw["train"], raw["validation"]


def normalize_boolq_example(ex: Dict[str, Any], min_valid_choices: int):
    passage = str(ex.get("passage", "")).strip()
    question = str(ex.get("question", "")).strip()
    answer = ex.get("answer", False)

    if question and not question.endswith("?"):
        question = question + "?"

    q_text = (
        f"Task: Answer the yes/no question using the passage.\n"
        f"Passage: {passage}\n"
        f"Question: {question}"
    )

    choice_texts = ["no", "yes"]
    label = 1 if bool(answer) else 0

    return q_text, choice_texts, label, passage, question


# ============================================================
# CACHE
# ============================================================

def cache_file_path(cfg: InferenceConfig, split_name: str) -> str:
    safe_ds = cfg.dataset_name.replace("/", "_")
    ensure_dir(cfg.cache_dir)

    return os.path.join(
        cfg.cache_dir,
        f"{safe_ds}_{split_name}_tok{cfg.chunk_tok_len}_seq{cfg.seq_len}.pt",
    )


def build_or_load_cached_split(
    cfg: InferenceConfig,
    split_name: str,
    hf_split,
    conceptizer: Optional[DebertaConceptizer],
):
    path = cache_file_path(cfg, split_name)

    if os.path.exists(path):
        print(f"[cache] loading {path}")
        return torch.load(path, map_location="cpu")

    if not cfg.build_cache_if_missing:
        raise FileNotFoundError(path)

    if conceptizer is None:
        raise RuntimeError("Conceptizer is required because cache is missing.")

    print(f"[cache] building {cfg.dataset_name}/{split_name}")

    rows = []
    skipped = 0

    for ex in tqdm(hf_split, desc=f"cache:{split_name}"):
        try:
            q_text, choice_texts, label, passage, question = normalize_boolq_example(
                ex,
                cfg.min_valid_choices,
            )

            q_seq, q_pad = conceptizer.encode_text_fixed(q_text)

            yes_variants = [
                f"Passage: {passage}\nQuestion: {question}\n{t}"
                for t in BOOLQ_YES_TEMPLATES
            ]
            no_variants = [
                f"Passage: {passage}\nQuestion: {question}\n{t}"
                for t in BOOLQ_NO_TEMPLATES
            ]

            all_choice_groups = []
            all_mask_groups = []

            for variants in [no_variants, yes_variants]:
                seqs = []
                pads = []

                for txt in variants:
                    s, p = conceptizer.encode_text_fixed(txt)
                    seqs.append(s)
                    pads.append(p)

                all_choice_groups.append(torch.stack(seqs, dim=0))
                all_mask_groups.append(torch.stack(pads, dim=0))

            rows.append(
                {
                    "q": q_seq,
                    "qmask": q_pad,
                    "choices": torch.stack(all_choice_groups, dim=0),  # [2, V, T, D]
                    "cmask": torch.stack(all_mask_groups, dim=0),      # [2, V, T]
                    "choice_mask": torch.tensor([True, True], dtype=torch.bool),
                    "label": int(label),
                    "num_choices": 2,
                }
            )

        except Exception as e:
            skipped += 1
            print(f"[warn] skipped one example: {type(e).__name__}: {e}")

    torch.save(rows, path)
    print(f"[cache] saved {path} ({len(rows)} examples, skipped={skipped})")

    return rows


# ============================================================
# DATASET
# ============================================================

class CachedMCQDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        r = self.rows[idx]

        return {
            "q": r["q"],
            "qmask": r["qmask"],
            "choices": r["choices"],
            "cmask": r["cmask"],
            "choice_mask": r["choice_mask"],
            "label": torch.tensor(r["label"], dtype=torch.long),
            "num_choices": r["num_choices"],
            "idx": idx,
        }


def cached_collate(batch):
    B = len(batch)
    T = batch[0]["q"].size(0)
    D = batch[0]["q"].size(1)
    K = batch[0]["choices"].size(0)
    V = batch[0]["choices"].size(1)

    q = torch.stack([item["q"] for item in batch], dim=0)
    qmask = torch.stack([item["qmask"] for item in batch], dim=0)

    choices = torch.zeros(B, K, V, T, D, dtype=torch.float32)
    cmask = torch.ones(B, K, V, T, dtype=torch.bool)
    choice_mask = torch.zeros(B, K, dtype=torch.bool)

    labels = torch.zeros(B, dtype=torch.long)
    idxs = torch.zeros(B, dtype=torch.long)

    for i, item in enumerate(batch):
        choices[i] = item["choices"]
        cmask[i] = item["cmask"]
        choice_mask[i] = item["choice_mask"]
        labels[i] = item["label"]
        idxs[i] = item["idx"]

    return {
        "q": q,
        "qmask": qmask,
        "choices": choices,
        "cmask": cmask,
        "choice_mask": choice_mask,
        "label": labels,
        "idx": idxs,
    }


# ============================================================
# MODEL
# ============================================================

def build_hlcm_from_cfg(cfg: InferenceConfig):
    return HyperbolicLCM(
        in_dim=cfg.in_dim,
        model_dim=cfg.model_dim,
        num_heads=cfg.num_heads,
        num_layers=cfg.num_layers,
        ffn_mult=cfg.ffn_mult,
        dropout=cfg.dropout,
        manifold_c=cfg.manifold_c,
        causal=cfg.causal,
        input_scale=cfg.input_scale,
        input_max_norm=cfg.input_max_norm,
    )


def load_grpo_best_model(cfg: InferenceConfig, device: torch.device):
    if not os.path.exists(cfg.grpo_ckpt_path):
        raise FileNotFoundError(cfg.grpo_ckpt_path)

    obj = torch.load(cfg.grpo_ckpt_path, map_location="cpu")

    if "model" not in obj:
        raise KeyError("grpo_best.pt does not contain key 'model'.")

    model = build_hlcm_from_cfg(cfg).to(device)

    missing, unexpected = model.load_state_dict(obj["model"], strict=False)

    print(f"[load] loaded model from {cfg.grpo_ckpt_path}")
    print(f"[load] stage: {obj.get('stage', 'unknown')}")
    print(f"[load] best_eval_acc: {obj.get('best_eval_acc', 'unknown')}")
    print(f"[load] epoch: {obj.get('epoch', 'unknown')}")
    print(f"[load] global_opt_step: {obj.get('global_opt_step', 'unknown')}")
    print(f"[load] missing keys: {len(missing)}")
    print(f"[load] unexpected keys: {len(unexpected)}")

    model.eval()

    for p in model.parameters():
        p.requires_grad = False

    return model, obj


# ============================================================
# LOGITS
# ============================================================

@torch.no_grad()
def hlcm_last_tangent(model, x, pad_mask, mu, sigma):
    device = next(model.parameters()).device

    x = x.to(device, non_blocking=True)
    pad_mask = pad_mask.to(device, non_blocking=True)

    if mu is not None and sigma is not None:
        x = (x - mu.view(1, 1, -1)) / sigma.view(1, 1, -1)

    h = model(x)
    h_tan = model.manifold.logmap0(h)

    B, T, D = h_tan.shape
    out = torch.empty((B, D), device=h_tan.device, dtype=h_tan.dtype)

    for i in range(B):
        valid = (~pad_mask[i]).nonzero(as_tuple=False).view(-1)
        j = int(valid[-1].item()) if valid.numel() else 0
        out[i] = h_tan[i, j]

    return out


@torch.no_grad()
def mcq_logits_hlcm(model, batch, mu, sigma, cfg):
    device = next(model.parameters()).device

    q = batch["q"].to(device, non_blocking=True)
    qmask = batch["qmask"].to(device, non_blocking=True)

    choices = batch["choices"].to(device, non_blocking=True)
    cmask = batch["cmask"].to(device, non_blocking=True)

    choice_mask = batch["choice_mask"].to(device, non_blocking=True)

    B, K, V, T, D = choices.shape

    e_q = hlcm_last_tangent(model, q, qmask, mu=mu, sigma=sigma)
    e_q = F.normalize(e_q, dim=-1)

    logits_per_choice = []

    for k in range(K):
        ch = choices[:, k]
        ch_mask = cmask[:, k]

        flat = ch.reshape(B * V, T, D)
        flat_mask = ch_mask.reshape(B * V, T)

        e_c = hlcm_last_tangent(model, flat, flat_mask, mu=mu, sigma=sigma)
        e_c = e_c.reshape(B, V, -1)
        e_c = F.normalize(e_c, dim=-1)

        sims = torch.einsum("bd,bvd->bv", e_q, e_c)
        choice_logit = sims.mean(dim=1)

        logits_per_choice.append(choice_logit)

    logits = torch.stack(logits_per_choice, dim=1)
    logits = logits / max(cfg.mcq_logit_temperature, 1e-6)

    logits = logits.masked_fill(~choice_mask, torch.finfo(logits.dtype).min)

    return logits


# ============================================================
# INFERENCE + TIMING
# ============================================================

@torch.no_grad()
def run_inference_with_time(model, loader, mu, sigma, cfg, save_path=None):
    model.eval()
    device = next(model.parameters()).device

    total = 0
    correct = 0
    total_loss = 0.0
    predictions = []

    if device.type == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

    t0 = time.perf_counter()

    for batch in tqdm(loader, desc=f"Inference BoolQ-{cfg.split}"):
        logits = mcq_logits_hlcm(
            model=model,
            batch=batch,
            mu=mu,
            sigma=sigma,
            cfg=cfg,
        )

        labels = batch["label"].to(logits.device, non_blocking=True)

        loss = F.cross_entropy(logits, labels)
        preds = logits.argmax(dim=-1)

        bs = labels.size(0)

        total += int(bs)
        correct += int((preds == labels).sum().item())
        total_loss += float(loss.item()) * int(bs)

        for i in range(bs):
            predictions.append(
                {
                    "example_index": int(batch["idx"][i].item()),
                    "gold": int(labels[i].item()),
                    "pred": int(preds[i].item()),
                    "correct": int(preds[i].item() == labels[i].item()),
                }
            )

    if device.type == "cuda":
        torch.cuda.synchronize()

    inference_time_sec = time.perf_counter() - t0

    results = {
        "num_examples": total,
        "correct": correct,
        "loss": total_loss / max(total, 1),
        "accuracy": correct / max(total, 1),
        "accuracy_percent": 100.0 * correct / max(total, 1),
        "inference_time_sec": inference_time_sec,
        "inference_time_hms": fmt_hms(inference_time_sec),
        "time_per_example_sec": inference_time_sec / max(total, 1),
        "examples_per_second": total / max(inference_time_sec, 1e-9),
        "predictions": predictions,
    }

    if save_path is not None:
        with open(save_path, "w") as f:
            json.dump(results, f, indent=2)
        print(f"[save] results -> {save_path}")

    return results


# ============================================================
# MAIN NOTEBOOK FUNCTION
# ============================================================

def inference_only_grpo_boolq(cfg: InferenceConfig):
    set_seed(cfg.seed)

    device = pick_device(cfg.prefer_gpu_index)

    if device.type == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.set_float32_matmul_precision("high")

    print("[device]", device)
    print("[checkpoint]", cfg.grpo_ckpt_path)

    ensure_dir(cfg.out_dir)
    ensure_dir(cfg.cache_dir)

    train_hf, val_hf = load_boolq_local(cfg)

    if cfg.split == "train":
        hf_split = train_hf
        split_name = "train"
    elif cfg.split in {"validation", "val", "dev"}:
        hf_split = val_hf
        split_name = "validation"
    else:
        raise ValueError("BoolQ local only has train and validation in this setup.")

    cache_path = cache_file_path(cfg, split_name)
    cache_build_time_sec = 0.0

    if not os.path.exists(cache_path):
        print(f"[cache] not found: {cache_path}")

        if not cfg.build_cache_if_missing:
            raise FileNotFoundError(cache_path)

        conceptizer = DebertaConceptizer(
            model_name=cfg.encoder_name,
            chunk_tok_len=cfg.chunk_tok_len,
            seq_len=cfg.seq_len,
            batch_size=cfg.encoder_batch_size,
            device=torch.device(cfg.conceptizer_device),
        )

        t_cache = time.perf_counter()

        rows = build_or_load_cached_split(
            cfg=cfg,
            split_name=split_name,
            hf_split=hf_split,
            conceptizer=conceptizer,
        )

        cache_build_time_sec = time.perf_counter() - t_cache

        del conceptizer
        cuda_cleanup()

    else:
        rows = build_or_load_cached_split(
            cfg=cfg,
            split_name=split_name,
            hf_split=hf_split,
            conceptizer=None,
        )

    print(f"[data] split={split_name}, examples={len(rows)}")
    print(f"[cache] {cache_path}")

    ds = CachedMCQDataset(rows)

    loader = DataLoader(
        ds,
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=cached_collate,
        drop_last=False,
    )

    model, ckpt_obj = load_grpo_best_model(cfg, device)
    mu, sigma = load_normalizer(cfg.normalizer_path, device)

    out_dir = os.path.join(cfg.out_dir, cfg.dataset_name)
    ensure_dir(out_dir)

    save_path = os.path.join(out_dir, f"grpo_best_inference_{split_name}_results.json")

    results = run_inference_with_time(
        model=model,
        loader=loader,
        mu=mu,
        sigma=sigma,
        cfg=cfg,
        save_path=save_path,
    )

    summary = {
        "dataset": cfg.dataset_name,
        "split": split_name,
        "checkpoint": cfg.grpo_ckpt_path,
        "checkpoint_stage": ckpt_obj.get("stage", None),
        "checkpoint_best_eval_acc": ckpt_obj.get("best_eval_acc", None),
        "checkpoint_epoch": ckpt_obj.get("epoch", None),
        "checkpoint_global_opt_step": ckpt_obj.get("global_opt_step", None),
        "cache_file": cache_path,
        "num_examples": results["num_examples"],
        "correct": results["correct"],
        "loss": results["loss"],
        "accuracy": results["accuracy"],
        "accuracy_percent": results["accuracy_percent"],
        "cache_build_time_sec": cache_build_time_sec,
        "cache_build_time_hms": fmt_hms(cache_build_time_sec),
        "inference_time_sec": results["inference_time_sec"],
        "inference_time_hms": results["inference_time_hms"],
        "time_per_example_sec": results["time_per_example_sec"],
        "examples_per_second": results["examples_per_second"],
    }

    summary_path = os.path.join(out_dir, f"grpo_best_inference_{split_name}_summary.json")

    with open(summary_path, "w") as f:
        json.dump(summary, f, indent=2)

    print("\n==================== GRPO BEST INFERENCE DONE ====================")
    print(json.dumps(summary, indent=2))
    print(f"[summary saved] {summary_path}")

    return summary, results

In [2]:
cfg = InferenceConfig(
    grpo_ckpt_path="runs/hlcm_boolq_local_grpo_better/boolq_local/grpo_best.pt",
    normalizer_path="normalizer.pt",

    hf_train_file="/home/user/twovolume/Nisha/Finetune/boolq/train-00000-of-00001.parquet",
    hf_validation_file="/home/user/twovolume/Nisha/Finetune/boolq/validation-00000-of-00001.parquet",

    out_dir="runs/hlcm_boolq_local_grpo_better",
    cache_dir="boolq_local_cached_features_grpo_better",

    split="validation",
    eval_batch_size=4,

    prefer_gpu_index=0,
    build_cache_if_missing=True,
)

summary, results = inference_only_grpo_boolq(cfg)

[device] cuda:0
[checkpoint] runs/hlcm_boolq_local_grpo_better/boolq_local/grpo_best.pt
[cache] not found: boolq_local_cached_features_grpo_better/boolq_local_validation_tok256_seq12.pt


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[cache] building boolq_local/validation


cache:validation: 100%|█████████████████████████████████████████| 3270/3270 [19:32<00:00,  2.79it/s]


[cache] saved boolq_local_cached_features_grpo_better/boolq_local_validation_tok256_seq12.pt (3270 examples, skipped=0)
[data] split=validation, examples=3270
[cache] boolq_local_cached_features_grpo_better/boolq_local_validation_tok256_seq12.pt
[load] loaded model from runs/hlcm_boolq_local_grpo_better/boolq_local/grpo_best.pt
[load] stage: grpo_best
[load] best_eval_acc: 0.6128440366972477
[load] epoch: 1
[load] global_opt_step: 590
[load] missing keys: 0
[load] unexpected keys: 0
[normalizer] loaded from normalizer.pt


Inference BoolQ-validation: 100%|█████████████████████████████████| 818/818 [02:50<00:00,  4.81it/s]


[save] results -> runs/hlcm_boolq_local_grpo_better/boolq_local/grpo_best_inference_validation_results.json

==================== GRPO BEST INFERENCE DONE ====================
{
  "dataset": "boolq_local",
  "split": "validation",
  "checkpoint": "runs/hlcm_boolq_local_grpo_better/boolq_local/grpo_best.pt",
  "checkpoint_stage": "grpo_best",
  "checkpoint_best_eval_acc": 0.6128440366972477,
  "checkpoint_epoch": 1,
  "checkpoint_global_opt_step": 590,
  "cache_file": "boolq_local_cached_features_grpo_better/boolq_local_validation_tok256_seq12.pt",
  "num_examples": 3270,
  "correct": 2004,
  "loss": 0.6810222802176753,
  "accuracy": 0.6128440366972477,
  "accuracy_percent": 61.28440366972477,
  "cache_build_time_sec": 1173.5123473447748,
  "cache_build_time_hms": "00:19:33",
  "inference_time_sec": 170.30190432583913,
  "inference_time_hms": "00:02:50",
  "time_per_example_sec": 0.05208009306600585,
  "examples_per_second": 19.201194566465325
}
[summary saved] runs/hlcm_boolq_local_grp

In [3]:
print("Accuracy:", summary["accuracy_percent"])
print("Inference time:", summary["inference_time_sec"])
print("Time/example:", summary["time_per_example_sec"])
print("Examples/sec:", summary["examples_per_second"])

Accuracy: 61.28440366972477
Inference time: 170.30190432583913
Time/example: 0.05208009306600585
Examples/sec: 19.201194566465325
